In [16]:
# =====================================================================
# Dual-View Mammography Deep Learning Framework
# 05_multitask_final.ipynb — Multi-Task Evaluation & Production Pipeline
#
# Core Focus: Load checkpoints dynamically if exists, bypassing training loops.
# T1_Char uses a safe state_dict key mapper compatible with linear classification layers.
# Optimal Fusion: Global Average Pooling (GAP) Raw Feature Fusion [MammoXV]
# Methodology: Strict Leakage-Free 5-Fold Nested Cross-Validation Matrix
#
# Triple Evaluation Tracks:
#   T1 — Characterization: Benign vs Malignant          (N=664)
#   T2 — Detection:        Malignant vs Normal+Benign   (N=961)
#   T3 — Screening:        Malignant vs Normal          (N=664)
# =====================================================================

import os
import random
import warnings
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

warnings.filterwarnings('ignore')

# ─── 1. GLOBAL CONFIGURATION & HYPERPARAMETERS ──────────────────────
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")

# Standardized publication-grade production parameters
MODEL_ID = 'eva02_base_patch14_448.mim_in22k_ft_in22k_in1k'
IMG_SIZE = 448       
BATCH_SIZE = 16      
NUM_EPOCHS = 50      
WARMUP_EPOCHS = 3    
ES_START = 8         
PATIENCE = 12        
NUM_WORKERS = 4
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Repository directory structures
SPLIT_CSV = './splits/patient_folds.csv'
IMAGE_DIR = './processed'  
ABL_CKPT_DIR = './checkpoints/ablation_suite'  
CKPT_DIR  = './checkpoints/all_tasks'
STAT_DIR  = './outputs/statistics'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(STAT_DIR, exist_ok=True)

# ─── 2. DATASET LAYER & METRIC TRANSFORMS ────────────────────────────
def get_transforms():
    train_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-10, 10), shear=(-5, 5), p=0.5),
        A.OneOf([A.ElasticTransform(alpha=30, sigma=5, p=1.0),
                 A.GridDistortion(num_steps=5, distort_limit=0.1, p=1.0)], p=0.3),
        A.OneOf([A.RandomBrightnessContrast(0.15, 0.15, p=1.0),
                 A.RandomGamma(gamma_limit=(85, 115), p=1.0)], p=0.5),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.CoarseDropout(max_holes=4, max_height=40, max_width=40, fill_value=0, p=0.3),
        A.Normalize(mean=MEAN, std=STD), 
        ToTensorV2(),
    ])
    val_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
        A.Normalize(mean=MEAN, std=STD), 
        ToTensorV2(),
    ])
    return train_tf, val_tf

class DualViewDataset(Dataset):
    def __init__(self, df, transform, label_col):
        self.df = df.reset_index(drop=True)
        self.tf = transform
        self.col = label_col
        
    def __len__(self): 
        return len(self.df)
        
    def _load(self, p):
        full_path = os.path.join(IMAGE_DIR, p)
        img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
        if img is None: 
            raise FileNotFoundError(f"Missing image slice path: {full_path}")
        return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            'cc':    self.tf(image=self._load(r['cc_path']))['image'],
            'mlo':   self.tf(image=self._load(r['mlo_path']))['image'],
            'label': int(r[self.col])
        }

# ─── 3. CLINICAL LOSS ARCHITECTURE (FOCAL LOSS MODULE) ────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.a = alpha
        self.g = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        return ((self.a * targets + (1 - self.a) * (1 - targets)) * (1 - pt)**self.g * bce).mean()

focal_criterion = FocalLoss()

# ─── 4. MODEL TOPOLOGIES & WRAPPERS ───────────────────────────────────
class MammoXVGAPFusionModel(nn.Module):
    def __init__(self, backbone_id=MODEL_ID, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_id, pretrained=pretrained, num_classes=0)
        dim = self.backbone.num_features
        self.head_bin = nn.Sequential(
            nn.Linear(dim * 2, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, cc, mlo):
        f_cc = self.backbone(cc)   
        f_mlo = self.backbone(mlo) 
        f_fused = torch.cat([f_cc, f_mlo], dim=-1) 
        return {'logit_bin': self.head_bin(f_fused).squeeze(-1)}

class MammoXVAblationInferenceWrapper(nn.Module):
    def __init__(self, backbone_id=MODEL_ID):
        super().__init__()
        self.backbone = timm.create_model(backbone_id, pretrained=False, num_classes=0)
        dim = self.backbone.num_features
        self.classifier = nn.Linear(dim * 2, 1)
    def forward(self, cc, mlo):
        f_cc = self.backbone(cc)
        f_mlo = self.backbone(mlo)
        f_fused = torch.cat([f_cc, f_mlo], dim=-1)
        return {'logit_bin': self.classifier(f_fused).squeeze(-1)}

# ─── 5. MIXED-MODE EXECUTION CRADLE (INFERENCE & TRAINING) ────────────
def run_production_fold(df_task, label_col, fold_k, task_name, task_mode='train'):
    train_tf, val_tf = get_transforms()
    df_outer_test = df_task[df_task['fold'] == fold_k].reset_index(drop=True)
    te_ld = DataLoader(DualViewDataset(df_outer_test, val_tf, label_col), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    ckpt_path = None
    is_ablation_file = False
    
    if task_mode == 'inference_only':
        search_pattern = os.path.join(ABL_CKPT_DIR, f"*No_Attention_GAP*fold{fold_k}.pt")
        candidate_paths = glob.glob(search_pattern)
        if not candidate_paths:
            search_pattern_alt = os.path.join(ABL_CKPT_DIR, f"*no_attention_gap*fold{fold_k}.pt")
            candidate_paths = glob.glob(search_pattern_alt)
            
        if candidate_paths:
            ckpt_path = candidate_paths[0]
            is_ablation_file = True
    else:
        ckpt_path = os.path.join(CKPT_DIR, f"production_{task_name}_fold{fold_k}.pt")

    if ckpt_path and os.path.exists(ckpt_path):
        print(f"  [CHECKPOINT FOUND] Loading weights from: {ckpt_path}")
        if is_ablation_file:
            model = MammoXVAblationInferenceWrapper().to(DEVICE)
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE), strict=False)
        else:
            model = MammoXVGAPFusionModel().to(DEVICE)
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE), strict=True)
            
        model.eval()
        te_p, te_y = [], []
        with torch.no_grad():
            for b in te_ld:
                o = model(b['cc'].to(DEVICE), b['mlo'].to(DEVICE))
                te_p.extend(torch.sigmoid(o['logit_bin']).cpu().numpy())
                te_y.extend(b['label'].numpy())
        return df_outer_test['patient_id'].values, np.array(te_y), np.array(te_p)
    
    if task_mode == 'inference_only':
        raise FileNotFoundError(f"Critical upstream ablation checkpoint missing at: {ABL_CKPT_DIR}")
    
    print(f"  [WARN] Checkpoint missing! Initializing training pipeline for {task_name} Fold {fold_k}")
    df_outer_train = df_task[df_task['fold'] != fold_k].copy()
    available_folds = [f for f in range(5) if f != fold_k]
    inner_val_fold = available_folds[-1]  
    
    df_inner_val   = df_outer_train[df_outer_train['fold'] == inner_val_fold].reset_index(drop=True)
    df_inner_train = df_outer_train[df_outer_train['fold'] != inner_val_fold].reset_index(drop=True)
    
    train_labels = df_inner_train[label_col].values
    class_counts = np.bincount(train_labels, minlength=2)
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)
    
    tr_ld = DataLoader(DualViewDataset(df_inner_train, train_tf, label_col), BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_ld = DataLoader(DualViewDataset(df_inner_val, val_tf, label_col), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = MammoXVGAPFusionModel().to(DEVICE)
    optimizer_head = torch.optim.AdamW([p for n, p in model.named_parameters() if 'backbone' not in n], lr=1e-4, weight_decay=1e-2)
    optimizer_bb   = torch.optim.AdamW(model.backbone.parameters(), lr=1e-5, weight_decay=1e-2)
    scheduler_bb   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_bb, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7)
    scaler = GradScaler()
    
    best_inner_auc, patience_ctr = -1.0, 0
    
    for epoch in range(1, NUM_EPOCHS + 1):
        frozen = epoch <= WARMUP_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad = not frozen
        model.train(); running_loss = 0.0
        for batch in tr_ld:
            optimizer_head.zero_grad(); optimizer_bb.zero_grad()
            with autocast():
                out = model(batch['cc'].to(DEVICE), batch['mlo'].to(DEVICE))
                loss = focal_criterion(out['logit_bin'], batch['label'].float().to(DEVICE))
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_head)
            if not frozen: scaler.unscale_(optimizer_bb)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_head)
            if not frozen: scaler.step(optimizer_bb)
            scaler.update(); running_loss += loss.item()
        if not frozen: scheduler_bb.step()
            
        model.eval(); va_p, va_y = [], []
        with torch.no_grad():
            for b in va_ld:
                o = model(b['cc'].to(DEVICE), b['mlo'].to(DEVICE))
                va_p.extend(torch.sigmoid(o['logit_bin']).cpu().numpy())
                va_y.extend(b['label'].numpy())
        inner_va_auc = roc_auc_score(va_y, va_p) if len(np.unique(va_y)) > 1 else 0.0
        
        improved = inner_va_auc > best_inner_auc
        if improved:
            best_inner_auc = inner_va_auc; patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
        elif epoch >= ES_START: patience_ctr += 1
        
        status = 'Warmup' if frozen else 'Active'
        marker = '*' if improved else ' '
        print(f'    Epoch {epoch:02d}[{status}] loss={running_loss/len(tr_ld):.4f} | Inner Val AUC={inner_va_auc:.4f} | Best Inner={best_inner_auc:.4f}{marker} | Pat={patience_ctr}')
        if epoch >= ES_START and patience_ctr >= PATIENCE: break
            
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval(); te_p, te_y = [], []
    with torch.no_grad():
        for b in te_ld:
            o = model(b['cc'].to(DEVICE), b['mlo'].to(DEVICE))
            te_p.extend(torch.sigmoid(o['logit_bin']).cpu().numpy())
            te_y.extend(b['label'].numpy())
    del model; torch.cuda.empty_cache()
    return df_outer_test['patient_id'].values, te_y, te_p

# ─── 6. DOWNSTREAM METRIC ANALYSIS MODULES ────────────────────────────
def compute_clinical_operating_points(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    # Locate the absolute optimal decision threshold using Youden's J Index matrix
    idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[idx]
    
    preds = (y_prob >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    return sens, spec

def bootstrap_auc_ci(y_true, y_pred):
    rng = np.random.default_rng(SEED); aucs = []
    for _ in range(1000):
        idx = rng.integers(0, len(y_true), len(y_true))
        if 0 < y_true[idx].sum() < len(idx): aucs.append(roc_auc_score(y_true[idx], y_pred[idx]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

# ─── 7. CORE PRODUCTION EXECUTION SUITE ──────────────────────────────
if __name__ == "__main__":
    df_main = pd.read_csv(SPLIT_CSV)
    df_main['label_char'] = df_main.apply(lambda r: 1 if r['class_3']=='Malign' else (0 if r['class_3']=='Benign' else -1), axis=1)
    df_main['label_det'] = (df_main['class_3'] == 'Malign').astype(int)
    df_main['label_screen'] = df_main.apply(lambda r: 1 if r['class_3']=='Malign' else (0 if r['class_3']=='Normal' else -1), axis=1)

    TASK_CONFIGS = [
        {"name": "T1_Char",   "label_col": "label_char",   "filter": lambda df: df[df['label_char']   != -1].copy(), "mode": "inference_only", "desc": "T1 Characterization (Ablation Locked)"},
        {"name": "T2_Det",    "label_col": "label_det",    "filter": lambda df: df.copy(),                            "mode": "train",          "desc": "T2 Detection (Production Checkpoint)"},
        {"name": "T3_Screen", "label_col": "label_screen", "filter": lambda df: df[df['label_screen'] != -1].copy(), "mode": "train",          "desc": "T3 Screening (Production Checkpoint)"},
    ]

    final_summary_rows = []
    for tc in TASK_CONFIGS:
        name, label_col = tc['name'], tc['label_col']
        df_task = tc['filter'](df_main).reset_index(drop=True)
        print(f"\n{'='*80}\n PROCESSING PARADIGM TRACK: {tc['desc']}\n{'='*80}")
        
        all_pids, all_ys, all_ps = [], [], []
        fold_aucs = []
        
        for fold_k in range(5):
            pids, ys, ps = run_production_fold(df_task, label_col, fold_k, name, tc['mode'])
            all_pids.extend(pids); all_ys.extend(ys); all_ps.extend(ps)
            
            if len(np.unique(ys)) > 1:
                fold_aucs.append(roc_auc_score(ys, ps))
            
        all_ys, all_ps = np.array(all_ys), np.array(all_ps)
        global_oof_auc = roc_auc_score(all_ys, all_ps)
        ci_lo, ci_hi = bootstrap_auc_ci(all_ys, all_ps)
        
        # Calculate summary parameters across individual cross-validation loops
        mean_fold_auc = np.mean(fold_aucs) if fold_aucs else 0.0
        std_fold_auc = np.std(fold_aucs) if fold_aucs else 0.0
        
        # Compute dynamic clinical Sensitivity and Specificity operational splits safely
        global_sens, global_spec = compute_clinical_operating_points(all_ys, all_ps)
        
        pd.DataFrame({'patient_id': all_pids, 'label': all_ys, 'prob': all_ps}).to_csv(os.path.join(CKPT_DIR, f"{name}_oof_final.csv"), index=False)
        
        final_summary_rows.append({
            'Task Clinical Track': name, 
            'Global OOF-AUC': f"{global_oof_auc:.4f}",
            'Mean Fold-AUC': f"{mean_fold_auc:.4f} (±{std_fold_auc:.4f})",
            '95% Confidence Interval': f"[{ci_lo:.4f} - {ci_hi:.4f}]",
            'Sensitivity': f"{global_sens:.4f}",
            'Specificity': f"{global_spec:.4f}"
        })
        
    df_final_matrix = pd.DataFrame(final_summary_rows)
    print("\n" + "="*135)
    print(" PERFORMANCE MATRIX GENERATED (ALL SCORES EXTRACTED DIRECTLY FROM CHECKPOINTS)")
    print("="*135)
    print(df_final_matrix.to_string(index=False))
    print("="*135)

Device: cuda | GPU: NVIDIA GeForce RTX 5090

 PROCESSING PARADIGM TRACK: T1 Characterization (Ablation Locked)
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/ablation_suite/ablation_t1_char_Ablation_No_Attention_GAP_fold0.pt
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/ablation_suite/ablation_t1_char_Ablation_No_Attention_GAP_fold1.pt
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/ablation_suite/ablation_t1_char_Ablation_No_Attention_GAP_fold2.pt
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/ablation_suite/ablation_t1_char_Ablation_No_Attention_GAP_fold3.pt
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/ablation_suite/ablation_t1_char_Ablation_No_Attention_GAP_fold4.pt

 PROCESSING PARADIGM TRACK: T2 Detection (Production Checkpoint)
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/all_tasks/production_T2_Det_fold0.pt
  [CHECKPOINT FOUND] Loading weights from: ./checkpoints/all_tasks/production_T2_Det_fold1.pt
  [CHECKPOINT F